<a href="https://colab.research.google.com/github/hsuancheyang/115-1-AI_Fundamentals/blob/main/%E5%B0%8D%E8%A9%B1%E6%A9%9F%E5%99%A8%E4%BA%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 設定 Google AI Studio API 金鑰

你需要一個 API 金鑰才能使用 Gemini API。如果你沒有，請在 [Google AI Studio](https://aistudio.google.com/app/apikey) 中建立一個金鑰。

在 Colab 中，您可以點擊左側面板中的 "鑰匙" 圖標，將金鑰儲存到秘密管理器中。請將其命名為 `GOOGLE_API_KEY`。然後，您可以像下面程式那樣將金鑰傳遞給 SDK：

In [12]:
# 導入 Python SDK
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

## 2. 初始化 Gemini 模型

在進行任何 API 呼叫之前，你需要初始化生成模型。我們將使用 `gemini-2.5-flash` 模型。

In [13]:
gemini_model = genai.GenerativeModel('models/gemini-2.5-flash')

## 3. 建立網頁版對話機器人介面 (使用 Gradio)

我們使用 `gradio` 函式庫來建立簡單的網頁介面，可輸入系統提示`system_prompt`（設定機器人人設）、調整 `top_p` (p 值) 和 `temperature` (回應溫度)，進行多輪對話。

首先，安裝 `gradio`：

In [14]:
!pip install -q gradio

接下來是聊天機器人的程式碼。它包含一個 `chat_session` 函數，用於處理對話邏輯，以及一個 `gradio` 介面來展示它。

In [15]:
import gradio as gr
import google.generativeai as genai

def chat_session(user_message, history, system_prompt, top_p, temperature):
    global gemini_model
    generation_config = {
        "temperature": temperature,
        "top_p": top_p,
    }

    model_history_for_chat = []
    if system_prompt:
        model_history_for_chat.append({'role': 'user', 'parts': [system_prompt]})
        model_history_for_chat.append({'role': 'model', 'parts': ['好的，我已收到指示。']})

    # -------- 修正這裡：因應新版 history 格式為字典列表 --------
    for message in history:
        # message 會是一個字典，例如 {'role': 'user', 'content': '...'}
        role = message['role']
        content = message['content']
        # 轉換成 Gemini SDK 要求的格式
        gemini_role = 'user' if role == 'user' else 'model'
        model_history_for_chat.append({'role': gemini_role, 'parts': [content]})
    # ----------------------------------------------------

    chat = gemini_model.start_chat(history=model_history_for_chat)
    try:
        response = chat.send_message(user_message, generation_config=generation_config)
        return response.text
    except Exception as e:
        print(f"Error sending message: {e}")
        return f"發生錯誤: {e}"

# Define initial values for reset
INITIAL_SYSTEM_PROMPT = ""
INITIAL_TOP_P = 0.9
INITIAL_TEMPERATURE = 0.7

with gr.Blocks(theme="soft", title="AI 聊天機器人助理") as demo:
    gr.Markdown("# 我的聊天機器人")
    gr.Markdown("你可以設定系統提示(當作機器人的人格設定)、調整選字策略 (top_p) 和回應溫度 (temperature)。")

    with gr.Row():
        with gr.Column(scale=1):
            system_prompt_input = gr.Textbox(
                label="系統提示 (System Prompt)",
                placeholder="設定聊天機器人的人設，例如：你是一個樂觀的AI助理。",
                lines=3,
                value=INITIAL_SYSTEM_PROMPT
            )
            top_p_slider = gr.Slider(
                minimum=0.0, maximum=1.0, step=0.01,
                value=INITIAL_TOP_P, label="選字策略 (Top P)",
                info="調整詞彙選擇的多樣性。值越大，選擇越多樣"
            )
            temperature_slider = gr.Slider(
                minimum=0.0, maximum=1.0, step=0.01,
                value=INITIAL_TEMPERATURE, label="回應溫度 (Temperature)",
                info="調整回應的隨機性。值越低會產生更可預測的回應，值越高回應的越熱情。"
            )
            clear_all_btn = gr.Button("清除所有 (Clear All)")

        with gr.Column(scale=2):
            chatbot_component = gr.Chatbot(height=300)
            msg = gr.Textbox(placeholder="輸入訊息...", container=False, scale=7)
            send_btn = gr.Button("送出")

# Define the response function for messages
def respond(message, chat_history, system_prompt, top_p, temperature):
    # 呼叫原始的 chat_session 函數
    bot_message = chat_session(message, chat_history, system_prompt, top_p, temperature)

    # -------- 修正這裡：改用符合新版格式的字典附加到歷史紀錄中 --------
    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "model", "content": bot_message})
    # -----------------------------------------------------------

    return chat_history

    # Clear function
    def clear_all_states_func():
        return (
            [], # clear chatbot history
            INITIAL_SYSTEM_PROMPT, # reset system prompt
            INITIAL_TOP_P, # reset top_p
            INITIAL_TEMPERATURE, # reset temperature
            ""   # clear message textbox
        )

    clear_all_btn.click(
        clear_all_states_func,
        outputs=[chatbot_component, system_prompt_input, top_p_slider, temperature_slider, msg]
    )

    # Setup interaction for message submission
    msg.submit(
        respond,
        [msg, chatbot_component, system_prompt_input, top_p_slider, temperature_slider],
        chatbot_component,
    ).then(
        lambda: gr.update(value=""),
        inputs=None,
        outputs=msg,
    )

    send_btn.click(
        respond,
        [msg, chatbot_component, system_prompt_input, top_p_slider, temperature_slider],
        chatbot_component,
    ).then(
        lambda: gr.update(value=""),
        inputs=None,
        outputs=msg,
    )

demo.launch(debug=True, share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f65badd9e68e2c6f80.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 